In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [2]:
load_dotenv()

True

In [3]:
model=ChatOpenAI()

In [4]:
MAX_TOKENS = 150

In [5]:
def call_model(state: MessagesState):

    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [7]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [8]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Mouryagna."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 11
Hi, my name is Mouryagna.


'Hello Mouryagna, nice to meet you! How can I assist you today?'

In [9]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 42
Hi, my name is Mouryagna.
Hello Mouryagna, nice to meet you! How can I assist you today?
I am learning LangGraph.


"That's great to hear! LangGraph is a fascinating topic. If you have any questions or need help with anything related to LangGraph, feel free to ask!"

In [10]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 98
Hi, my name is Mouryagna.
Hello Mouryagna, nice to meet you! How can I assist you today?
I am learning LangGraph.
That's great to hear! LangGraph is a fascinating topic. If you have any questions or need help with anything related to LangGraph, feel free to ask!
Can you explain short term memory?


'Short-term memory is a type of memory that temporarily stores information for a short period of time, typically lasting for a few seconds to a few minutes. It is also known as working memory.\n\nShort-term memory is responsible for holding information that is currently being used or processed in cognitive tasks, such as remembering a phone number before dialing it or following instructions in a recipe. It allows us to retain and manipulate small amounts of data for immediate use.\n\nShort-term memory capacity is limited and can only hold a certain amount of information at once. If the information is not rehearsed or transferred to long-term memory, it is quickly forgotten. This is why short-term memory is often described as a temporary and fragile storage system.\n\nOverall, short-term memory plays a crucial role in our daily cognitive functions by allowing us to hold onto and manipulate information temporarily until it is either forgotten or transferred to long-term memory for more p

In [11]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


"I'm sorry, but I do not have access to that information."

In [12]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Mouryagna.
------------------------------------------------------------------------------------------------------------------------
Hello Mouryagna, nice to meet you! How can I assist you today?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
That's great to hear! LangGraph is a fascinating topic. If you have any questions or need help with anything related to LangGraph, feel free to ask!
------------------------------------------------------------------------------------------------------------------------
Can you explain short term memory?
------------------------------------------------------------------------------------------------------------------------
Short-term memory is a type of memory that temporarily stores information for a short period of t